# Lab 9 — Background Tasks for Batch AI Processing

**Difficulty: Beginner | ~30 min | Requires Lab 2 (Async/Await)**

### Step 0: Install Dependencies

This cell installs every pinned dependency the lab needs. Run this first so all later cells have what they require.

In [2]:
!pip install fastapi==0.112.2 pydantic==2.8.2 httpx==0.28.1 python-dotenv==1.2.3 openai==3.5.0 uvicorn==0.30.6


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### Step 1: Imports, API Key, and App Setup

The imports bring in everything we need: FastAPI for the endpoints, `BackgroundTasks` so the endpoint can hand off work and return immediately, the OpenAI client for calling OpenRouter's LLM, and the standard-library modules for timing, threading, and unique job IDs. The API key is loaded from the `.env` file the same way as in earlier labs.

In [3]:
from fastapi import FastAPI, BackgroundTasks
from pydantic import BaseModel
from dotenv import load_dotenv
from openai import AsyncOpenAI
import httpx, uvicorn, os, time, uuid, threading

load_dotenv()
api_key = os.getenv("OPEN_ROUTER_KEY") or input("Open Router API key: ")
client = AsyncOpenAI(api_key=api_key, base_url="https://openrouter.ai/api/v1")
app = FastAPI()

### Step 2: Job Store and ID Generator

A module-level dictionary acts as our shared state — both the POST endpoint (which creates jobs) and the background function (which updates them) read and write to it. This is the bridge between the request handler, which returns immediately, and the background work, which happens later. A small helper generates a short, unique hex ID for each job.

In [4]:
job_store: dict[str, dict] = {}

def generate_job_id():
    return uuid.uuid4().hex

### Step 3: Summarize a Single Review

This helper makes one LLM call to summarize a single review in one or two sentences. It exists separately from the batch loop so each step of the pipeline — calling the LLM, handling the response — is visible on its own.

In [5]:
async def summarize_single_review(review_text):
    response = await client.chat.completions.create(
        model="openrouter/free",
        messages=[{"role": "user", "content": f"Summarize this customer review in one or two sentences: {review_text}"}],
    )
    return response.choices[0].message.content

### Step 4: The Background Task

This is the function that runs **after** the POST response has already been returned. It sets the job status to `running`, loops through each review one at a time (each review gets its own real LLM call), and after each summary finishes it appends the result and increments the completed counter. Because it updates `job_store` after every single item, a poll made mid-run shows partial progress — you can watch completed climb from 0 to 1 to 2 while the job is still going.

When `simulate_failure` is True, the function raises an exception on the second review. The try/except catches it, marks the job as `failed`, and stores a clear error message — but leaves the already-completed items in place. No rollback happens.

In [ ]:
async def run_batch_summary(job_id, reviews, simulate_failure=False):
    job_store[job_id]["status"] = "running"
    try:
        for index, review in enumerate(reviews):
            if simulate_failure and index == 1:
                raise RuntimeError("Simulated failure on review 2")
            summary = await summarize_single_review(review)
            job_store[job_id]["results"].append(summary)
            job_store[job_id]["completed"] += 1
        job_store[job_id]["status"] = "done"
        
    except Exception as exc:
        job_store[job_id]["status"] = "failed"
        job_store[job_id]["error"] = str(exc)

### Step 5: The POST and GET Endpoints

The POST endpoint creates a new job entry in `job_store`, schedules `run_batch_summary` as a background task via FastAPI's `BackgroundTasks`, and returns the job ID and status **immediately** — the background task has not run at all yet when this response is constructed, which is the whole point.

The GET endpoint simply looks up the job by ID and returns whatever state it finds — no waiting, no blocking. The client polls this endpoint whenever it wants to check progress.

In [7]:
class BatchRequest(BaseModel):
    reviews: list[str]
    simulate_failure: bool = False

@app.post("/tasks/summarize-batch")
async def summarize_batch(req: BatchRequest, background_tasks: BackgroundTasks):
    job_id = generate_job_id()
    job_store[job_id] = {"status": "pending", "total": len(req.reviews), "completed": 0, "results": [], "error": None}
    background_tasks.add_task(run_batch_summary, job_id, req.reviews, req.simulate_failure)
    return {"job_id": job_id, "status": "pending", "total": len(req.reviews)}

@app.get("/tasks/{job_id}")
async def get_task(job_id: str):
    return job_store.get(job_id, {"error": f"Job {job_id} not found"})

### Step 6: Serve the App on a Real Server

The demos need to hit real HTTP endpoints, so we run uvicorn in a background daemon thread and create a real `httpx.Client`. The 404 from the root path confirms the server is up.

In [ ]:
PORT = 8778
def run_server():
    uvicorn.run(app, host="127.0.0.1", port=PORT, log_level="warning")
    
threading.Thread(target=run_server, daemon=True).start()
time.sleep(2)
http_client = httpx.Client(timeout=180)
print("Server up:", http_client.get(f"http://127.0.0.1:{PORT}/").status_code)

Server up: 404


### Demo 1: Timing Proof

This is the core demonstration: the POST response returns in a fraction of a second, long before any LLM call has happened. We time the POST call with `time.perf_counter()` and print the immediate response. The elapsed time should be very small — that is the point.

In [ ]:
reviews_batch = [
    "This product exceeded my expectations. The build quality is fantastic and it arrived quickly.",
    "Not worth the money. The item broke after just two weeks of regular use.",
    "Decent product for the price. Does what it says, nothing more.",
]

start = time.perf_counter()
res = http_client.post(f"http://127.0.0.1:{PORT}/tasks/summarize-batch", json={"reviews": reviews_batch})
elapsed = time.perf_counter() - start

response_data = res.json()
demo1_job_id = response_data["job_id"]

print(f"Elapsed: {elapsed:.4f}s")
print(f"Response: {response_data}")
print("No LLM call has happened yet — the response returned before any work started.")

Elapsed: 0.0041s
Response: {'job_id': '730b06eb249b4be5bf24e216dbf3aa80', 'status': 'pending', 'total': 3}
No LLM call has happened yet — the response returned before any work started.


### Demo 2: Polling Over Time

Now we poll the GET endpoint a few times with short waits between polls. Each poll prints the full state — you should see `completed` increase as reviews finish being summarized, eventually reaching `done` with all results.

In [ ]:
poll = 0
while True:
    state = http_client.get(f"http://127.0.0.1:{PORT}/tasks/{demo1_job_id}").json()
    poll += 1
    print(f"Poll {poll}: status={state['status']}, completed={state['completed']}/{state['total']}")
    
    if state["status"] == "done":
        break
    time.sleep(4)

print("\nResults:")
for i, summary in enumerate(state["results"], 1):
    print(f"  {i}. {summary}")

Poll 1: status=running, completed=2/3
Poll 2: status=done, completed=3/3

Results:
  1. The product exceeded the customer's expectations thanks to its fantastic build quality and fast shipping.
  2. The customer considers the product a failure, stating it broke within two weeks of regular use and is therefore not worth the money.
  3. 

Of course. Here are a few options:

**Option 1 (Most direct):** The customer found the product to be decent and functional for its price, but noted it offers no extra features beyond what is advertised.

**Option 2 (More concise):** This review describes the product as a basic, no-frills item that meets its stated purpose without exceeding expectations.


### Demo 3: Two Independent Jobs

We submit two different batches back-to-back, then poll both to completion. This confirms that each job's results correspond to its own input reviews and are never mixed up with the other job's.

In [11]:
batch_a = ["Amazing customer service experience.", "Delivery was late but product is fine."]
batch_b = ["Terrible quality, completely unusable.", "Perfect fit, highly recommend to others."]

job_a = http_client.post(f"http://127.0.0.1:{PORT}/tasks/summarize-batch", json={"reviews": batch_a}).json()["job_id"]
job_b = http_client.post(f"http://127.0.0.1:{PORT}/tasks/summarize-batch", json={"reviews": batch_b}).json()["job_id"]

print(f"Job A: {job_a}\nJob B: {job_b}")

while True:
    state_a = http_client.get(f"http://127.0.0.1:{PORT}/tasks/{job_a}").json()
    state_b = http_client.get(f"http://127.0.0.1:{PORT}/tasks/{job_b}").json()

    if state_a["status"] == "done" and state_b["status"] == "done":
        break
    time.sleep(4)
    
print("\nJob A (customer service / delivery reviews):")
for i, s in enumerate(state_a["results"], 1): print(f"  {i}. {s}")
print("\nJob B (quality / recommendation reviews):")
for i, s in enumerate(state_b["results"], 1): print(f"  {i}. {s}")
print("\nEach job's results match its own input — no mixing.")

Job A: 6abaec2d10d048a8a2eba9210d68eb8d
Job B: 3096fdae49834fcfa2614c826faccb5b

Job A (customer service / delivery reviews):
  1. The reviewer had an amazing customer service experience.
  2. The delivery was late, but the product arrived in good condition and met expectations.

Job B (quality / recommendation reviews):
  1. The reviewer is extremely dissatisfied, describing the product as "terrible quality" and "completely unusable." Overall, they express strong disappointment and do not recommend it.
  2. The reviewer says the product is a perfect fit and strongly recommends it to others.

Each job's results match its own input — no mixing.


### Demo 4: Controlled Failure

We submit a batch with `simulate_failure=True`. The background task will process the first review successfully, then raise an exception on the second review. We poll until the status becomes `failed` and inspect the final state: completed should reflect only the items that finished before the failure, and error should contain a clear message.

In [12]:
fail_batch = ["Great product, works as expected.", "This one should fail.", "This one never gets processed."]
res = http_client.post(f"http://127.0.0.1:{PORT}/tasks/summarize-batch", json={"reviews": fail_batch, "simulate_failure": True})
fail_job_id = res.json()["job_id"]

while True:
    state = http_client.get(f"http://127.0.0.1:{PORT}/tasks/{fail_job_id}").json()
    if state["status"] in ("done", "failed"): break
    time.sleep(2)
    
print(f"Final state: status={state['status']}, completed={state['completed']}/{state['total']}")
print(f"Results ({len(state['results'])} items):")
for i, s in enumerate(state["results"], 1): print(f"  {i}. {s}")
print(f"Error: {state['error']}")
print("Only 1 review completed before the failure. The process stopped at index 1.")

Final state: status=failed, completed=1/3
Results (1 items):
  1. The customer is satisfied with the product and confirms it meets their expectations.
Error: Simulated failure on review 2
Only 1 review completed before the failure. The process stopped at index 1.
